# Inżynieria cech - PM2.5 z pogodą

Milestone 2, zadanie 2: złączenie oczyszczonego PM2.5 z pogodą Open-Meteo i policzenie cech pochodnych pod model godzinowego PM2.5. Wynik trafia do `data/silver/pm25_pogoda_2024.parquet`.

Jeden wiersz = jedna stacja w jednej godzinie (UTC). Funkcje są w `scripts/silver_features.py`.

## 1. Wczytanie danych z silver

In [5]:
from pathlib import Path
import runpy

import polars as pl

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
BRONZE_DIR = ROOT / 'data' / 'bronze'
SILVER_DIR = ROOT / 'data' / 'silver'

feat = runpy.run_path(str(ROOT / 'scripts' / 'silver_features.py'))

measurements = pl.read_parquet(SILVER_DIR / 'pomiary_2024.parquet')
weather = pl.read_parquet(SILVER_DIR / 'openmeteo_2024.parquet')
stations = pl.read_parquet(BRONZE_DIR / 'stacje.parquet')

print('pomiary:', measurements.shape)
print('pogoda: ', weather.shape)

pomiary: (8472637, 9)
pogoda:  (658800, 12)


## 2. Złączenie PM2.5 z pogodą

Szkieletem jest pogoda, bo ma pełną siatkę godzin (96 stacji × 8784 h). PM2.5 dołączamy przez `LEFT JOIN` po `kod_stacji` + `data_czas_utc`, więc godzina bez pomiaru ma `pm25 = NULL`, a nie znika. Dzięki temu `shift(k)` zawsze oznacza k godzin wstecz, także tam, gdzie czyszczenie usunęło pomiary.

Z metadanych stacji bierzemy `typ_stacji` i `typ_obszaru`. Przed złączeniem usuwamy spację z kodu `LuLubsStrzelMOB ` (8.17) i duplikat `OpKrap3Maja` (8.2).

In [6]:
base = feat['build_base'](measurements, weather, stations)

pm25_rows = measurements.filter((pl.col('wskaznik') == 'PM2.5') & (pl.col('czas_usredniania') == '1g')).height

assert base.height == weather.height
assert base.select(pl.struct('kod_stacji', 'data_czas_utc').is_duplicated().any()).item() is False
assert base.group_by('kod_stacji').len()['len'].unique().to_list() == [8784]
assert base['typ_stacji'].null_count() == 0

print('wiersze:', base.height, '| z pomiarem PM2.5:', base['pm25'].is_not_null().sum(), 'z', pm25_rows)

wiersze: 658800 | z pomiarem PM2.5: 635919 z 816760


86 pomiarów PM2.5 nie trafia do tabeli: to pierwsza godzina 2024 r. czasu polskiego (2023-12-31 23:00 UTC), dla której nie pobieraliśmy pogody.

## 3. Cechy

| Grupa | Cechy | Uzasadnienie |
|---|---|---|
| kalendarz | `godzina`, `dzien_tygodnia`, `miesiac`, `weekend`, `swieto`, `dzien_wolny` | ruch i ogrzewanie mają rytm dobowy i tygodniowy; liczone w czasie polskim, bo za nim idzie aktywność ludzi |
| kalendarz cykliczny | `godzina_sin/cos`, `miesiac_sin/cos` | 23:00 jest blisko 0:00, a grudzień blisko stycznia |
| historia PM2.5 | `pm25_lag1/2/3/24/168` | stężenie zmienia się powoli; lag 24 h i 168 h łapie rytm doby i tygodnia |
| okna PM2.5 | `pm25_srednia3h/6h/24h`, `pm25_max24h` | wygładzona historia, odporna na pojedyncze braki |
| wiatr | `wiatr_u`, `wiatr_v` | składowe wektora zamiast kąta: 359° i 1° to prawie ten sam kierunek; przy ciszy oba 0 |
| pogoda pochodna | `cisnienie_zmiana3h`, `temperatura_zmiana24h`, `opad_suma6h/24h` | front atmosferyczny przewietrza, opad wymywa pył |
| ogrzewanie | `stopniodni_grzania` = max(0, 15 °C − temperatura) | im zimniej, tym więcej spalania w domach |
| stacja | `typ_stacji`, `typ_obszaru`, `wysokosc_m`, `siatka_szerokosc/dlugosc` | stacja komunikacyjna w mieście ma inny poziom niż tło poza miastem |

Zmienna docelowa to `pm25` w godzinie t. Historia PM2.5 kończy się na t-1, więc model z tymi cechami prognozuje na godzinę do przodu. Pogoda z godziny t to dane rzeczywiste z reanalizy, czyli zakładamy, że pogodę na tę godzinę znamy (w praktyce z prognozy).

Okna liczymy z `min_samples=1`: średnia z dostępnych godzin, NULL dopiero gdy w całym oknie nie ma pomiaru.

In [7]:
features = feat['add_features'](base)

print(features.shape)
features.filter(pl.col('kod_stacji') == 'DsWrocAlWisn').select(
    'data_czas_utc', 'godzina', 'pm25', 'pm25_lag1', 'pm25_srednia3h',
    'wind_speed_10m', 'wind_direction_10m', 'wiatr_u', 'wiatr_v', 'stopniodni_grzania',
).slice(30, 6)

(658800, 40)


data_czas_utc,godzina,pm25,pm25_lag1,pm25_srednia3h,wind_speed_10m,wind_direction_10m,wiatr_u,wiatr_v,stopniodni_grzania
datetime[μs],i8,f64,f64,f64,f64,f64,f64,f64,f64
2024-01-02 06:00:00,7,13.5,10.9,8.933333,8.0,216.0,4.702282,6.472136,13.6
2024-01-02 07:00:00,8,10.8,13.5,11.133333,8.1,193.0,1.822104,7.892398,13.3
2024-01-02 08:00:00,9,12.8,10.8,11.733333,11.3,187.0,1.377124,11.215772,11.9
2024-01-02 09:00:00,10,8.8,12.8,12.366667,13.4,172.0,-1.86492,13.269592,10.9
2024-01-02 10:00:00,11,7.2,8.8,10.8,12.9,153.0,-5.856477,11.493984,10.4
2024-01-02 11:00:00,12,10.0,7.2,9.6,14.0,157.0,-5.470236,12.887068,9.7


## 4. Kontrola wyniku

Najważniejsze: żadna cecha nie może widzieć bieżącej wartości PM2.5 (wyciek danych). Sprawdzamy, że lag 1 to wartość z poprzedniej godziny tej samej stacji, a okna kończą się na t-1. Pierwsze godziny każdej stacji nie mogą brać wartości z poprzedniej stacji.

In [8]:
by_station = features.sort('kod_stacji', 'data_czas_utc')
prev = by_station.with_columns(
    pl.col('pm25').shift(1).over('kod_stacji').alias('prev'),
    pl.col('pm25').shift(1).rolling_mean(3, min_samples=1).over('kod_stacji').alias('mean3'),
)

assert prev.filter(pl.col('pm25_lag1').ne_missing(pl.col('prev'))).height == 0
assert prev.filter((pl.col('pm25_srednia3h') - pl.col('mean3')).abs() > 1e-9).height == 0
assert features.filter(pl.col('data_czas_utc') == pl.col('data_czas_utc').min())['pm25_lag1'].null_count() == 96
assert features.filter(pl.col('wind_speed_10m') == 0).select((pl.col('wiatr_u') == 0) & (pl.col('wiatr_v') == 0)).to_series().all()
assert features['stopniodni_grzania'].min() >= 0

# Ostatnia godzina UTC należy lokalnie do Nowego Roku 2025.
last = features.filter(pl.col('data_czas_utc') == pl.col('data_czas_utc').max())
assert last['swieto'].all() and last['dzien_wolny'].all()

features.null_count().transpose(include_header=True, column_names=['braki']).filter(pl.col('braki') > 0)


AssertionError: 

Braki w cechach PM2.5 wynikają z luk w pomiarach. Zostawiamy je w silver - wybór, czy usuwać wiersze, czy uzupełniać braki, należy do etapu gold i zależy od modelu.

Szybki podgląd siły cech - korelacja Pearsona z `pm25` (tylko zależności liniowe, więc to wskazówka, nie selekcja):

In [ ]:
numeric = [c for c, t in features.schema.items() if t.is_numeric() and c != 'pm25']

(
    features
    .select(pl.corr('pm25', c).alias(c) for c in numeric)
    .transpose(include_header=True, header_name='cecha', column_names=['korelacja'])
    .sort(pl.col('korelacja').abs(), descending=True)
)

cecha,korelacja
str,f64
"""pm25_lag1""",0.949606
"""pm25_srednia3h""",0.910581
"""pm25_lag2""",0.888574
"""pm25_srednia6h""",0.854091
"""pm25_lag3""",0.829271
…,…
"""miesiac""",-0.025619
"""godzina""",0.02428
"""godzina_sin""",-0.011644


## 5. Zapis do silver

In [ ]:
path = SILVER_DIR / 'pm25_pogoda_2024.parquet'
features.write_parquet(path)
print(f'{path.name:26} {features.height:>10,} wierszy  {features.width} kolumn  {path.stat().st_size / 1_000_000:5.1f} MB')

pm25_pogoda_2024.parquet      843,264 wierszy  40 kolumn   34.0 MB


## 6. Horyzont prognozy: 1 h i 24 h

Tabela nie zakłada jednego horyzontu. Zmienna docelowa to zawsze `pm25` w godzinie t, a horyzont wybieramy w gold przez zestaw cech: model może używać tylko tego, co było znane h godzin wcześniej. Oba modele trenujemy bezpośrednio (direct) na tej samej tabeli, a nie przez 24-krotne powtarzanie modelu 1 h (rekurencja kumuluje błąd).

| Grupa cech | 1 h do przodu | 24 h do przodu |
|---|---|---|
| `pm25_lag1`, `pm25_lag2`, `pm25_lag3` | tak | nie - wartości z ostatniej doby |
| `pm25_srednia3h/6h/24h`, `pm25_max24h` | tak | nie - okna kończą się na t-1 |
| `pm25_lag24`, `pm25_lag168` | tak | tak |
| pogoda z godziny t i cechy pogodowe | tak | tak |
| kalendarz i stacja | tak | tak |

Pogodę z godziny t traktujemy w obu wariantach jako znaną, czyli zakładamy prognozę pogody. Ponieważ to dane rzeczywiste z reanalizy, wynik modelu jest górnym oszacowaniem tego, co dałaby prawdziwa prognoza.

Model 1 h jest zdominowany przez `pm25_lag1` (korelacja 0,95 z `pm25`), więc w praktyce przewiduje „tyle co godzinę temu”. Model 24 h musi przewidzieć zmianę stężenia, więc pogoda ma w nim większą wagę. Oba porównujemy z baseline'em persystencji: `pm25_lag1` dla 1 h i `pm25_lag24` dla 24 h.

Możliwe rozszerzenie dla 24 h: średnie PM2.5 z okien kończących się na t-24 (np. t-48...t-25). Obecnie najbliższa historia dostępna przy tym horyzoncie to sam `pm25_lag24`.

## Dalej: partycjonowanie i benchmark

Uruchom `06_partycjonowanie_benchmark.ipynb`. Zapisuje całą tabelę cech w wariantach A/B, sprawdza zachowanie wszystkich kolumn i NULL-i oraz publikuje zweryfikowane partycje dla gold. Zachowujemy pełną siatkę UTC; przy zmianie czasu lokalna godzina może się powtarzać.
